# ODI to Databricks Migration: `insert_hr_trg_emp.sql`

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** This notebook converts an ODI SQL statement for loading `TRG_EMP` from `EMPLOYEES` into Databricks Spark SQL. It performs a direct insert operation.

In [ ]:
# Create ETL parameter widgets (as no specific parameters found in source, common ones are included)
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type");
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID");
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process WID");
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number");

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Load Target Table: `trg_emp`

In [ ]:
%sql
-- SCEN_TASK_NO {10}, {20}, {30}: These tasks are typically markers or pre-steps in ODI.
-- In Databricks, the focus is on the SQL execution.
-- The following INSERT statement loads data directly into the target table.
INSERT INTO workspace.hr.trg_emp
  (
    employee_id ,
    first_name ,
    last_name ,
    email ,
    phone_number ,
    hire_date ,
    job_id ,
    salary ,
    commission_pct ,
    manager_id ,
    department_id
  )
SELECT
  employees.employee_id ,
  employees.first_name ,
  employees.last_name ,
  employees.email ,
  employees.phone_number ,
  employees.hire_date ,
  employees.job_id ,
  employees.salary ,
  employees.commission_pct ,
  employees.manager_id ,
  employees.department_id
FROM
  workspace.hr.employees AS employees;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_emp
FROM workspace.hr.trg_emp;

## Conversion Notes and Manual Actions Required

1.  **Table Schema Definition:** The DDL for `HR.TRG_EMP` and `HR.EMPLOYEES` was not provided in the source. This notebook assumes that `workspace.hr.trg_emp` and `workspace.hr.employees` tables already exist in Databricks with compatible schemas. If not, DDL statements must be created and executed prior to running this notebook. Ensure Oracle `NUMBER` types map to `BIGINT` or `DECIMAL` appropriately, `VARCHAR2` to `STRING`, and `DATE` to `TIMESTAMP`.
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable to Databricks Delta Lake, which handles append and parallelism automatically.
3.  **Schema and Naming Convention:** All Oracle schema and table names (`HR.TRG_EMP`, `HR.EMPLOYEES`) have been converted to lowercase with the `workspace.` prefix (`workspace.hr.trg_emp`, `workspace.hr.employees`) as per the naming conventions.
4.  **ODI Parameters:** Common ETL parameters (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`) have been included as `dbutils` widgets. Although not explicitly used in this simple `INSERT` statement, they are standard for ODI migrations.
5.  **Error Handling:** The original ODI snippet did not include error handling. If error logging (`E$` tables) is required, separate cells for error table creation and population would need to be added.
6.  **Incremental Load:** This `INSERT` statement represents a full load or an initial load. If incremental loading is intended, a `WHERE` clause based on a last-loaded timestamp or other criteria would need to be added to the `SELECT` statement, potentially combined with a `MERGE` statement for updates/inserts.
7.  **Optimize:** An `OPTIMIZE` statement for `trg_emp` might be beneficial for performance if this table is frequently queried or used in joins. If applied, ensure `SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;` precedes it in the same cell.